#### 文本流输出

In [1]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from agent_lab.model_hub import ModelRegistry

model = init_chat_model(**ModelRegistry.DEEPSEEK_V4_FLASH)

In [2]:
full = None
for chunk in model.stream('what color is the sky.'):
    full = chunk if full is None else full + chunk
    print(full.text)


That
That's
That's a
That's a great
That's a great question
That's a great question!
That's a great question! The
That's a great question! The straightforward
That's a great question! The straightforward answer
That's a great question! The straightforward answer is
That's a great question! The straightforward answer is that
That's a great question! The straightforward answer is that the
That's a great question! The straightforward answer is that the sky
That's a great question! The straightforward answer is that the sky appears
That's a great question! The straightforward answer is that the sky appears **
That's a great question! The straightforward answer is that the sky appears **blue
That's a great question! The straightforward answer is that the sky appears **blue**
That's a great question! The straightforward answer is that the sky appears **blue** on
That's a great question! The straightforward answer is that the sky appears **blue** on a
That's a great question! The straightfor

In [3]:
type(full), type(chunk)

(langchain_core.messages.ai.AIMessageChunk,
 langchain_core.messages.ai.AIMessageChunk)

In [4]:
from langchain.messages import AIMessageChunk

assert isinstance(full, AIMessageChunk)
assert isinstance(chunk, AIMessageChunk)

full.content_blocks

[{'type': 'text',
  'text': 'That\'s a great question! The straightforward answer is that the sky appears **blue** on a clear, sunny day.\n\nHowever, the full answer is a bit more interesting:\n\n*   **Why blue?** It\'s due to a phenomenon called **Rayleigh scattering**. Sunlight is made up of all the colors of the rainbow. As it travels through Earth\'s atmosphere, it bumps into tiny gas molecules and particles. Blue light has a shorter wavelength and is scattered much more than other colors, so we see a blue sky from all directions.\n*   **Sunrise and Sunset:** The sky often appears **red, orange, and pink** at sunrise and sunset. This is because the sunlight has to travel through much more of the atmosphere to reach your eyes. The blue light gets scattered away, leaving the longer wavelengths of red and orange to dominate.\n*   **At Night:** Without the sun\'s direct light, the sky appears **black**, and we can see the stars.\n*   **Overcast Days:** When the sky is full of clouds, i

#### 工具调用流输出

In [7]:
reasoner = init_chat_model(**ModelRegistry.DEEPSEEK_V4_FLASH_THINKING)

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city"""
    return f"It's always sunny in {city}."

reasoner = reasoner.bind_tools([get_weather_for_location])

In [8]:
full = None
for chunk in reasoner.stream('what is the weather in sf?'):
    for block in chunk.content_blocks:
        if block['type'] == 'reasoning' and (reasoning := block.get('reasoning')):
            print(reasoning, end='')
        elif block['type'] == 'tool_call_chunk':
            print(block)
        elif block['type'] == 'text':
            print(block['text'], end='')
    full = chunk if full is None else full + chunk

The user is asking about the weather in San Francisco (sf). Let me get the weather for that location.{'type': 'tool_call_chunk', 'id': 'call_00_oV3OvEPjdCzqXNLNb1yZKzgV', 'name': 'get_weather_for_location', 'args': '', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '{', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': 'city', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': ': ', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': 'San', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': ' Francisco', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'na

In [9]:
type(full)

langchain_core.messages.ai.AIMessageChunk

In [10]:
assert isinstance(full, AIMessageChunk)

full.content_blocks

[{'type': 'reasoning',
  'reasoning': 'The user is asking about the weather in San Francisco (sf). Let me get the weather for that location.'},
 {'type': 'tool_call',
  'id': 'call_00_oV3OvEPjdCzqXNLNb1yZKzgV',
  'name': 'get_weather_for_location',
  'args': {'city': 'San Francisco'}}]